# 范围契约和任务边界

模型不知道什么时候工作停止。范围契约是单个工作的文件，告诉你工作从哪里开始，到哪里结束，以及溢出的时候怎么会滚。这个契约将“在这个范围内”从愿望变成了检查。

## 问题描述

Agents蔓延。任务是“修复登陆bug”，实际修改到了登陆路由、email工具、数据库驱动、README。

范围蔓延是最容易被监视忽略的错误模式，因为agent自信地讲述了每个步骤。修复方式不是靠更严格的提示词，而是靠磁盘上的契约，用来表明哪些被许可，以及检查修改结果是否与许可内容相斥。

## 基本概念
```mermaid
flowchart TD
A[Task]-->B[scope_contract.json]-->C[Agent loop]-->D[final diff]-->E[scope_checker.py]-->F[in scope?]--yes-->G[Verification Gate]
B-->E
F--no-->H[block + open question]
```

### 范围契约中发生了什么

|内容|目标|
|---|---|
|task_id|关联到看板上的任务|
|goal|审查者可以验证的一句话|
|allowed_files|agent可以修改的文件|
|forbidden_files|agent不能修改的文件|
|acceptance_criteria|表明工作完成的验证命令或者断言|
|rollback_plan|如果需要回退，怎么办|
|approvals_required|范围外的操作是否需要人为显式确认|

没有`forbidden_files`的契约是不完整的，负空间也构成契约的一半。

### 使用GLOBS而不是裸露的路径

实际仓库可能移动文件，契约中标记文件使用globs(app/*/.py)。从而会话间发生了重构不会导致契约失效。

### 回滚也是范围的一部分

列出怎么样回滚的同时也在强迫契约的作者思考哪些内容会导致失败。一个没有办法回滚的契约是不应该被批准的契约。

### 范围检查也是差异检查

主要是检查diff。

### 范围的两个层级：功能列表和任务契约

范围契约约束一个任务，而不会约束整个项目。一个agent能够完美的遵循契约修复登陆问题，并且在下一轮中，决定项目还需要一个设置页面、一个暗主题。契约从来没有要求过哪些项目工作在范围内，只有哪些文件在范围内。

这第二种层级需要它自己的原语：会话开始的时候agent读去的`feautre_list.json`文件。它是机器可读的、有序文件的项目后台日志。agent每次跳出一个状态为todo的功能，然后将它的id写到激活的范围契约里面，并且禁止相同会话开启第二个功能。“一次只做一个功能”不再是提示词中的一行，而是变成了可以从磁盘中读取的，可编码成硬性门禁的检查。

```json
{
    "project": "...",
    "active": "import ...",
    "features": [
        {"id": "import ...", "status": "in_progress"},
        {"id": "...", "status": "todo"}
    ]
}
```

# 开始编码

对应本章核心：**`scope_contract.json`（含 forbidden + rollback）**、**glob 差异检查**、**范围蔓延 → block**、**`feature_list.json` 一次只做一个功能**。  
玩具先跑通契约与 checker；生产段用 **LangChain 工具 + DeepSeek** 强制经门禁编辑。不硬凑 PyTorch。


## 1. 教学玩具：ScopeContract + FeatureList + ScopeChecker

- 契约缺 `forbidden_files` / `rollback_plan` 直接拒批。
- `check_paths` 对 diff 做 glob 命中；越界 / 命中 forbidden → 阻断。
- `feature_list`：已有 `active` 时拒绝第二个功能。


In [ ]:
from __future__ import annotations

import json
import shutil
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path, PurePosixPath
from typing import Any, Literal


FeatureStatus = Literal["todo", "in_progress", "done", "blocked"]


def match_glob(path: str, pattern: str) -> bool:
    """
    用 pathlib full_match 做 glob 匹配（支持 **）。

    Args:
        path: 仓库相对路径，POSIX 风格。
        pattern: glob，如 ``src/auth/**/*.py``。
    Returns:
        matched: 是否命中。
    """
    p = PurePosixPath(path.replace("\\", "/"))
    return p.full_match(pattern.replace("\\", "/"))


def any_glob(path: str, patterns: list[str]) -> bool:
    return any(match_glob(path, g) for g in patterns)


@dataclass
class ScopeContract:
    """单任务范围契约：许可空间 + 负空间 + 回滚。"""

    task_id: str
    goal: str
    allowed_files: list[str]
    forbidden_files: list[str]
    acceptance_criteria: list[str]
    rollback_plan: str
    approvals_required: bool = True
    feature_id: str | None = None

    def validate(self) -> None:
        """
        Raises:
            ValueError: 契约不完整（缺 forbidden / rollback 等）。
        """
        if not self.task_id or not self.goal:
            raise ValueError("task_id and goal required")
        if not self.allowed_files:
            raise ValueError("allowed_files required")
        if not self.forbidden_files:
            raise ValueError("forbidden_files required (negative space is half the contract)")
        if not self.rollback_plan.strip():
            raise ValueError("rollback_plan required; unrollbackable contracts must not be approved")
        if not self.acceptance_criteria:
            raise ValueError("acceptance_criteria required")

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "ScopeContract":
        c = cls(
            task_id=str(data["task_id"]),
            goal=str(data["goal"]),
            allowed_files=list(data.get("allowed_files", [])),
            forbidden_files=list(data.get("forbidden_files", [])),
            acceptance_criteria=list(data.get("acceptance_criteria", [])),
            rollback_plan=str(data.get("rollback_plan", "")),
            approvals_required=bool(data.get("approvals_required", True)),
            feature_id=data.get("feature_id"),
        )
        c.validate()
        return c


@dataclass
class Feature:
    id: str
    status: FeatureStatus = "todo"
    title: str = ""


@dataclass
class FeatureList:
    """项目级功能列表：一次只做一个功能。"""

    project: str
    active: str | None = None
    features: list[Feature] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {
            "project": self.project,
            "active": self.active,
            "features": [asdict(f) for f in self.features],
        }

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "FeatureList":
        feats = [Feature(**f) for f in data.get("features", [])]
        return cls(project=str(data.get("project", "")), active=data.get("active"), features=feats)

    def claim(self, feature_id: str) -> Feature:
        """
        认领一个 todo 功能；禁止同会话开启第二个。

        Args:
            feature_id: 功能 id。
        Returns:
            feature: 变为 in_progress 的功能。
        Raises:
            RuntimeError: 已有 active 或目标不可认领。
        """
        if self.active and self.active != feature_id:
            raise RuntimeError(f"one feature at a time: active={self.active}, refuse {feature_id}")
        for f in self.features:
            if f.id == feature_id:
                if f.status not in ("todo", "in_progress"):
                    raise RuntimeError(f"feature {feature_id} status={f.status}")
                f.status = "in_progress"
                self.active = feature_id
                return f
        raise RuntimeError(f"unknown feature {feature_id}")


@dataclass
class ScopeVerdict:
    ok: bool
    in_scope: list[str] = field(default_factory=list)
    out_of_scope: list[str] = field(default_factory=list)
    forbidden_hits: list[str] = field(default_factory=list)
    needs_approval: list[str] = field(default_factory=list)
    detail: str = ""


class ScopeChecker:
    """范围检查 = 差异检查：对照契约 globs 审查改动路径。"""

    def __init__(self, contract: ScopeContract) -> None:
        contract.validate()
        self.contract = contract

    def check_paths(self, changed: list[str]) -> ScopeVerdict:
        """
        Args:
            changed: diff 中的相对路径列表。
        Returns:
            verdict: 是否全部在范围内。
        """
        in_scope: list[str] = []
        out: list[str] = []
        forbidden: list[str] = []
        needs: list[str] = []
        for path in changed:
            norm = path.replace("\\", "/").lstrip("./")
            if any_glob(norm, self.contract.forbidden_files):
                forbidden.append(norm)
                continue
            if any_glob(norm, self.contract.allowed_files):
                in_scope.append(norm)
            else:
                out.append(norm)
                if self.contract.approvals_required:
                    needs.append(norm)

        ok = not forbidden and not out
        detail = "in scope" if ok else "block + open question"
        return ScopeVerdict(
            ok=ok,
            in_scope=in_scope,
            out_of_scope=out,
            forbidden_hits=forbidden,
            needs_approval=needs,
            detail=detail,
        )


class ScopeWorkbench:
    """磁盘上的两层范围：feature_list.json + scope_contract.json。"""

    def __init__(self, root: Path) -> None:
        self.root = Path(root)
        self.feature_path = self.root / "feature_list.json"
        self.contract_path = self.root / "scope_contract.json"
        self.diff_log = self.root / "changed_files.json"

    def save_features(self, fl: FeatureList) -> None:
        self.feature_path.write_text(json.dumps(fl.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    def load_features(self) -> FeatureList:
        return FeatureList.from_dict(json.loads(self.feature_path.read_text(encoding="utf-8")))

    def save_contract(self, c: ScopeContract) -> None:
        c.validate()
        self.contract_path.write_text(json.dumps(c.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    def load_contract(self) -> ScopeContract:
        return ScopeContract.from_dict(json.loads(self.contract_path.read_text(encoding="utf-8")))

    def record_change(self, path: str) -> list[str]:
        """
        记录一次编辑（模拟 diff 条目）。

        Args:
            path: 相对路径。
        Returns:
            changed: 累计改动列表。
        """
        changed: list[str] = []
        if self.diff_log.exists():
            changed = list(json.loads(self.diff_log.read_text(encoding="utf-8")))
        norm = path.replace("\\", "/").lstrip("./")
        if norm not in changed:
            changed.append(norm)
        self.diff_log.write_text(json.dumps(changed, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        return changed

    def check_diff(self) -> ScopeVerdict:
        changed = []
        if self.diff_log.exists():
            changed = list(json.loads(self.diff_log.read_text(encoding="utf-8")))
        return ScopeChecker(self.load_contract()).check_paths(changed)


def make_scope_repo() -> Path:
    """
    Returns:
        root: 带 feature_list 与示例文件的临时仓库。
    """
    root = Path(tempfile.mkdtemp(prefix="scope_task_"))
    (root / "src" / "auth").mkdir(parents=True)
    (root / "src" / "email").mkdir(parents=True)
    (root / "src" / "auth" / "login.py").write_text("def login():\n    return True\n", encoding="utf-8")
    (root / "src" / "email" / "send.py").write_text("def send():\n    pass\n", encoding="utf-8")
    (root / "README.md").write_text("# demo\n", encoding="utf-8")
    fl = FeatureList(
        project="demo-login",
        active=None,
        features=[
            Feature(id="FIX-LOGIN", status="todo", title="修复登陆校验"),
            Feature(id="DARK-THEME", status="todo", title="暗主题"),
        ],
    )
    wb = ScopeWorkbench(root)
    wb.save_features(fl)
    return root


print("scope task ready | contract + globs + feature_list gate")


## 2. 玩具示例

不完整契约拒绝、一次一功能、范围内通过、email/README 蔓延被挡、glob 覆盖新路径。


In [ ]:
def demo_scope_task() -> None:
    """契约完整性、glob 差异检查、一次一个功能。"""
    root = make_scope_repo()
    try:
        wb = ScopeWorkbench(root)

        # 不完整契约：缺 forbidden
        try:
            ScopeContract(
                task_id="T1",
                goal="fix login",
                allowed_files=["src/auth/**/*.py"],
                forbidden_files=[],
                acceptance_criteria=["pytest"],
                rollback_plan="git checkout -- src/auth",
            ).validate()
            raise AssertionError("expected missing forbidden")
        except ValueError as e:
            assert "forbidden" in str(e)
            print("incomplete contract rejected ok")

        # 无回滚不可批准
        try:
            ScopeContract(
                task_id="T1",
                goal="fix login",
                allowed_files=["src/auth/**/*.py"],
                forbidden_files=["README.md", "src/email/**"],
                acceptance_criteria=["pytest -q"],
                rollback_plan="  ",
            ).validate()
            raise AssertionError("expected rollback required")
        except ValueError as e:
            assert "rollback" in str(e)
            print("no-rollback contract rejected ok")

        fl = wb.load_features()
        fl.claim("FIX-LOGIN")
        wb.save_features(fl)
        try:
            fl.claim("DARK-THEME")
            raise AssertionError("expected one-feature gate")
        except RuntimeError as e:
            assert "one feature" in str(e)
            print("one feature at a time ok")

        contract = ScopeContract(
            task_id="T-LOGIN",
            goal="修复登陆校验边界",
            feature_id="FIX-LOGIN",
            allowed_files=["src/auth/**/*.py"],
            forbidden_files=["README.md", "src/email/**", "**/.env"],
            acceptance_criteria=["python -m pytest -q tests/test_login.py"],
            rollback_plan="git checkout -- src/auth && reopen FIX-LOGIN",
            approvals_required=True,
        )
        wb.save_contract(contract)

        # 范围内编辑
        wb.record_change("src/auth/login.py")
        v1 = wb.check_diff()
        assert v1.ok and "src/auth/login.py" in v1.in_scope
        print("in-scope diff ok")

        # 范围蔓延：email + README
        wb.record_change("src/email/send.py")
        wb.record_change("README.md")
        v2 = wb.check_diff()
        assert not v2.ok
        assert "src/email/send.py" in v2.forbidden_hits or "src/email/send.py" in v2.out_of_scope
        assert "README.md" in v2.forbidden_hits
        print("scope creep blocked ok")

        # glob 扛住路径形态：auth 下新文件仍在允许域
        only = ScopeChecker(contract).check_paths(["src/auth/validators.py"])
        assert only.ok
        print("glob survives refactor-shaped paths ok")

        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_scope_task()


## 3. 生产级：LangChain 范围门禁 + DeepSeek

工具：`claim_feature` / `edit_file` / `check_scope` / `approve_path`。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_WB: ScopeWorkbench | None = None
HUMAN_APPROVED: set[str] = set()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_prod_scope() -> str:
    """
    Returns:
        json: 新仓库 root。
    """
    global PROD_ROOT, PROD_WB, HUMAN_APPROVED
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    HUMAN_APPROVED = set()
    PROD_ROOT = make_scope_repo()
    PROD_WB = ScopeWorkbench(PROD_ROOT)
    # 预置契约（尚未认领功能）
    PROD_WB.save_contract(
        ScopeContract(
            task_id="T-LOGIN",
            goal="修复登陆校验边界",
            feature_id=None,
            allowed_files=["src/auth/**/*.py"],
            forbidden_files=["README.md", "src/email/**", "**/.env"],
            acceptance_criteria=["python -m pytest -q tests/test_login.py"],
            rollback_plan="git checkout -- src/auth && reopen FIX-LOGIN",
            approvals_required=True,
        )
    )
    return json.dumps({"root": str(PROD_ROOT)}, ensure_ascii=False)


def _wb() -> ScopeWorkbench:
    if PROD_WB is None:
        reset_prod_scope()
    assert PROD_WB is not None
    return PROD_WB


class EmptyArgs(BaseModel):
    pass


class FeatureArgs(BaseModel):
    feature_id: str = Field(description="如 FIX-LOGIN")


class PathArgs(BaseModel):
    path: str


class ApproveArgs(BaseModel):
    path: str = Field(description="人工批准的范围外路径")


def build_scope_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        return reset_prod_scope()

    def _list_features(**kwargs: Any) -> str:
        return json.dumps(_wb().load_features().to_dict(), ensure_ascii=False)

    def _claim(**kwargs: Any) -> str:
        fid = FeatureArgs(**kwargs).feature_id
        fl = _wb().load_features()
        try:
            feat = fl.claim(fid)
        except RuntimeError as e:
            return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)
        _wb().save_features(fl)
        c = _wb().load_contract()
        c.feature_id = fid
        _wb().save_contract(c)
        return json.dumps({"ok": True, "feature": asdict(feat), "contract_feature_id": fid}, ensure_ascii=False)

    def _load_contract(**kwargs: Any) -> str:
        return json.dumps(_wb().load_contract().to_dict(), ensure_ascii=False)

    def _edit(**kwargs: Any) -> str:
        path = PathArgs(**kwargs).path
        # 先检查单文件是否立刻命中 forbidden
        c = _wb().load_contract()
        norm = path.replace("\\", "/").lstrip("./")
        if any_glob(norm, c.forbidden_files):
            return json.dumps({"ok": False, "blocked": True, "reason": "forbidden_files", "path": norm}, ensure_ascii=False)
        if not any_glob(norm, c.allowed_files):
            if norm not in HUMAN_APPROVED and c.approvals_required:
                return json.dumps(
                    {"ok": False, "blocked": True, "reason": "out_of_scope_needs_approval", "path": norm},
                    ensure_ascii=False,
                )
        changed = _wb().record_change(norm)
        # 真正写一点内容，模拟编辑
        target = _wb().root / norm
        if target.exists() and target.is_file():
            target.write_text(target.read_text(encoding="utf-8") + f"\n# touched by scope agent\n", encoding="utf-8")
        return json.dumps({"ok": True, "changed": changed}, ensure_ascii=False)

    def _approve(**kwargs: Any) -> str:
        p = ApproveArgs(**kwargs).path.replace("\\", "/").lstrip("./")
        HUMAN_APPROVED.add(p)
        return json.dumps({"ok": True, "approved": sorted(HUMAN_APPROVED)}, ensure_ascii=False)

    def _check(**kwargs: Any) -> str:
        v = _wb().check_diff()
        return json.dumps(asdict(v), ensure_ascii=False)

    return [
        StructuredTool.from_function(name="reset_scope_repo", description="重建演示仓库。", func=_reset, args_schema=EmptyArgs),
        StructuredTool.from_function(name="list_features", description="读取 feature_list.json。", func=_list_features, args_schema=EmptyArgs),
        StructuredTool.from_function(name="claim_feature", description="认领一个 todo 功能（一次只能一个）。", func=_claim, args_schema=FeatureArgs),
        StructuredTool.from_function(name="load_scope_contract", description="读取 scope_contract.json。", func=_load_contract, args_schema=EmptyArgs),
        StructuredTool.from_function(name="edit_file", description="尝试编辑文件；forbidden/未批准越界会被拒。", func=_edit, args_schema=PathArgs),
        StructuredTool.from_function(name="approve_path", description="人工批准越界路径（演示 HITL）。", func=_approve, args_schema=ApproveArgs),
        StructuredTool.from_function(name="check_scope", description="对累计 diff 跑 scope_checker。", func=_check, args_schema=EmptyArgs),
    ]


SCOPE_TOOLS = build_scope_tools()


def build_scope_agent():
    """
    Returns:
        agent: 受范围契约约束的 control agent。
    """
    system = (
        "你是范围控制代理。先 list_features / claim_feature，再按 scope_contract 编辑。\n"
        "禁止改 forbidden；越界先停并说明，不要硬闯。一次只做一个功能。\n"
        "最后 check_scope。用中文简短说明。"
    )
    return create_agent(get_llm(), SCOPE_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 30) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:280]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:240]}")
            n += 1


print(f"scope task production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM；脚本路径仍验证蔓延阻断。


In [ ]:
def demo_prod_scope() -> None:
    """脚本化门禁 +（有 key 时）DeepSeek。"""
    reset_prod_scope()
    wb = _wb()

    fl = wb.load_features()
    fl.claim("FIX-LOGIN")
    wb.save_features(fl)
    c = wb.load_contract()
    c.feature_id = "FIX-LOGIN"
    wb.save_contract(c)

    wb.record_change("src/auth/login.py")
    assert wb.check_diff().ok

    wb.record_change("README.md")
    v = wb.check_diff()
    assert not v.ok and "README.md" in v.forbidden_hits
    print("=== scripted ===")
    print(json.dumps({"in_scope_ok": True, "creep_blocked": True}, ensure_ascii=False))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_scope_agent()
    prompt = (
        "reset_scope_repo，list_features，claim_feature FIX-LOGIN，load_scope_contract，"
        "edit_file src/auth/login.py，再尝试 edit_file src/email/send.py（应被拒），"
        "尝试 claim_feature DARK-THEME（应被拒），check_scope，中文说明范围契约如何挡住蔓延。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))

    fl2 = _wb().load_features()
    assert fl2.active == "FIX-LOGIN"
    verdict = _wb().check_diff()
    assert verdict.ok or (
        "src/auth/login.py" in verdict.in_scope and not verdict.forbidden_hits
    ) or "src/email" not in json.dumps(asdict(verdict))
    # email 若未被写入 diff，check 应 ok；若写入则应不 ok
    changed = []
    if _wb().diff_log.exists():
        changed = json.loads(_wb().diff_log.read_text(encoding="utf-8"))
    assert "src/email/send.py" not in changed
    assert "src/auth/login.py" in changed
    print("PROD DEMO OK")


demo_prod_scope()
